# MRI Brain Tumor Classifier: training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PasinduSuraweera/mri-brain-tumor-diagnosis-system/blob/upgrade-app/CNN_brain_tumor_detection.ipynb)

This notebook retrains the classifier behind the web app with the reproducible pipeline in
[`training/train.py`](training/train.py), then checks the result and exports it for the app.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Training takes roughly 15–30 minutes on a T4.

| Step | What happens |
|------|--------------|
| 1. Settings | choose the backbone, epochs, and where the data comes from |
| 2. Setup | clone the repo and install the two extra packages |
| 3. Data | download the [Kaggle Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) (or use your Drive copy) |
| 4. Train | de-duplicate, train, fine-tune, calibrate, evaluate, export |
| 5. Results | metrics and plots, compared with the previous VGG16 model |
| 6. App check | run the exported model through the web app's own inference code with Grad-CAM |
| 7. Save | sample scans for the app, backup to Drive, optional push to GitHub |

*The original VGG16 notebook that produced `models/model.h5` is preserved in the git history.*

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/PasinduSuraweera/mri-brain-tumor-diagnosis-system.git"
BRANCH = "upgrade-app"
REPO_DIR = "/content/mri-brain-tumor-diagnosis-system"

DATA_SOURCE = "kaggle"                                 # "kaggle" (automatic download) or "drive"
DRIVE_DATA_DIR = "/content/drive/MyDrive/MRI Images"   # used when DATA_SOURCE == "drive"; needs Training/ and Testing/

ARCH = "efficientnetv2b0"     # "efficientnetv2b0" (fast), "efficientnetv2b2" or "convnext_tiny" (bigger)
IMAGE_SIZE = 224
HEAD_EPOCHS = 8               # epochs with the backbone frozen
FINETUNE_EPOCHS = 30          # max fine-tuning epochs (early stopping usually ends sooner)
EXTRA_ARGS = []               # any other train.py options, e.g. ["--dedup-threshold", "-1"]

EXPORT_DIR = "models"         # the web app serves whatever models/model_meta.json points to
ADD_SAMPLE_SCANS = True       # put one test scan per class into static/samples (check the dataset licence first)
SAVE_TO_DRIVE = True          # back up the run and exported model to Google Drive
PUSH_TO_GITHUB = False        # commit + push the new model; needs a GITHUB_TOKEN Colab secret with repo write access
GIT_NAME, GIT_EMAIL = "PasinduSuraweera", "pasisuraweera2003@gmail.com"

## 2. Setup

In [ ]:
import glob
import json
import os
import random
import shutil
import subprocess
import sys
from datetime import datetime

import tensorflow as tf

IN_COLAB = "google.colab" in sys.modules
print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "none! Training will be very slow. Use Runtime → Change runtime type → GPU.")


def run(cmd, env=None):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, **(env or {})})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")


if not os.path.exists(os.path.join(REPO_DIR, "training", "train.py")):
    # GIT_LFS_SKIP_SMUDGE skips the 128 MB legacy model.h5, which training does not need.
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], env={"GIT_LFS_SKIP_SMUDGE": "1"})
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "ai-edge-litert==2.3.0", "kagglehub"])

## 3. Data

The Kaggle dataset is public, so `kagglehub` can usually download it without an account. If it asks for
credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (🔑 icon on the left) and re-run this cell.

In [ ]:
if DATA_SOURCE == "kaggle":
    import kagglehub

    if IN_COLAB:
        from google.colab import userdata

        for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass  # secret not set; anonymous download is used
    root = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
elif DATA_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    root = DRIVE_DATA_DIR
else:
    root = DATA_SOURCE  # a local folder

candidates = [os.path.dirname(p) for p in sorted(glob.glob(os.path.join(root, "**", "Training"), recursive=True))
              if os.path.isdir(os.path.join(os.path.dirname(p), "Testing"))]
if not candidates:
    raise FileNotFoundError(f"No folder with Training/ and Testing/ found under {root}")
DATA_DIR = candidates[0]
print("Data folder:", DATA_DIR)

CLASSES = sorted(d for d in os.listdir(os.path.join(DATA_DIR, "Training")) if not d.startswith("."))
print(f"{'class':<12}{'train':>8}{'test':>8}")
for c in CLASSES:
    n_train = len(os.listdir(os.path.join(DATA_DIR, "Training", c)))
    n_test = len(os.listdir(os.path.join(DATA_DIR, "Testing", c)))
    print(f"{c:<12}{n_train:>8}{n_test:>8}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

random.seed(0)
fig, axes = plt.subplots(2, len(CLASSES), figsize=(3 * len(CLASSES), 6))
for col, c in enumerate(CLASSES):
    files = sorted(glob.glob(os.path.join(DATA_DIR, "Training", c, "*")))
    for row, f in enumerate(random.sample(files, 2)):
        axes[row, col].imshow(Image.open(f).convert("RGB"))
        axes[row, col].set_title(c)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 4. Train

`train.py` removes duplicate and leaked images, trains the head, fine-tunes the backbone with early stopping,
calibrates the confidence scores on the validation split, evaluates on the untouched test split and exports the
model into `models/` for the web app.

In [ ]:
RUN_DIR = os.path.join(REPO_DIR, "training", "runs", datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + ARCH)
run([sys.executable, "training/train.py",
     "--data-dir", DATA_DIR,
     "--arch", ARCH,
     "--image-size", str(IMAGE_SIZE),
     "--head-epochs", str(HEAD_EPOCHS),
     "--finetune-epochs", str(FINETUNE_EPOCHS),
     "--run-dir", RUN_DIR,
     "--export-dir", EXPORT_DIR,
     *EXTRA_ARGS],
    env={"TF_CPP_MIN_LOG_LEVEL": "2"})

## 5. Results

Both models are scored on the same Kaggle `Testing/` split (de-duplication only removes *training* images).
The old VGG16 numbers were measured with augmentation applied to the test images, so they are only roughly comparable.

In [ ]:
from IPython.display import Image as ShowImage, display

metrics = json.load(open(os.path.join(RUN_DIR, "metrics.json")))
old = json.load(open("training/vgg16_metrics.json"))


def fmt(x, pct=False):
    return "–" if x is None else (f"{x * 100:.1f}%" if pct else f"{x:.3f}")


print(f"{'':<28}{'old VGG16':>12}{'new ' + ARCH:>22}")
print(f"{'Test accuracy':<28}{fmt(old['test_accuracy'], True):>12}{fmt(metrics['test_accuracy'], True):>22}")
print(f"{'Macro F1':<28}{'–':>12}{fmt(metrics['macro_f1']):>22}")
print(f"{'Macro ROC AUC':<28}{'–':>12}{fmt(metrics['macro_auc_ovr']):>22}")
print(f"{'Calibration error (ECE)':<28}{'–':>12}{fmt(metrics['ece_calibrated']):>22}"
      f"   (before temperature scaling: {fmt(metrics['ece_uncalibrated'])}, T = {metrics['temperature']:.2f})")
print(f"{'Confident predictions':<28}{'–':>12}{fmt(metrics['coverage_at_threshold'], True):>22}"
      f"   (accuracy on those: {fmt(metrics['accuracy_at_threshold'], True)})")
print()
print(f"{'Recall per class':<28}{'old':>12}{'new':>22}")
for c in metrics["per_class"]:
    print(f"{c:<28}{fmt(old['per_class'][c]['recall'], True):>12}{fmt(metrics['per_class'][c]['recall'], True):>22}")
print()
print("De-duplication:", metrics["deduplication"])

display(ShowImage(os.path.join(RUN_DIR, "training_curves.png")))
display(ShowImage(os.path.join(RUN_DIR, "confusion_matrix.png")))

## 6. Check the exported model through the web app's own code

This uses `inference.py` exactly as the web app does. It checks two things:
* the app's "is this an MRI?" input checks should **not** reject real test scans;
* accuracy and the share of *inconclusive* answers on the full test split after export.

In [ ]:
import importlib

import numpy as np

import inference

importlib.reload(inference)
clf = inference.Classifier(EXPORT_DIR)
print("Serving:", clf.meta["description"])

test_files = [(f, c) for c in clf.labels for f in sorted(glob.glob(os.path.join(DATA_DIR, "Testing", c, "*")))]
rejected, warned, correct, confident, confident_correct = [], 0, 0, 0, 0
for path, label in test_files:
    img = inference.open_image(open(path, "rb").read())
    reasons, warnings = inference.check_input(img)
    if reasons:
        rejected.append((path, reasons[0]))
    warned += bool(warnings)
    pred = clf.predict(img, with_heatmap=False)
    correct += pred["label"] == label
    confident += pred["confident"]
    confident_correct += pred["confident"] and pred["label"] == label

n = len(test_files)
print(f"Test scans: {n}")
print(f"Accuracy after export:          {correct / n:.2%}")
print(f"Inconclusive (below threshold):  {1 - confident / n:.2%}")
print(f"Accuracy when confident:         {confident_correct / max(confident, 1):.2%}")
print(f"Wrongly rejected by input check: {len(rejected)} ({len(rejected) / n:.2%})")
print(f"Shown a background warning:      {warned} ({warned / n:.2%})")
for path, reason in rejected[:10]:
    print("  rejected:", os.path.relpath(path, DATA_DIR), "→", reason)

In [ ]:
random.seed(1)
fig, axes = plt.subplots(2, len(clf.labels), figsize=(3.2 * len(clf.labels), 8))
for col, label in enumerate(clf.labels):
    files = [f for f, c in test_files if c == label]
    for row, path in enumerate(random.sample(files, 2)):
        img = inference.open_image(open(path, "rb").read())
        pred = clf.predict(img)
        overlay = inference.heatmap_overlay(inference.thumbnail(img, 512), pred["cam"])
        ax = axes[row, col]
        ax.imshow(overlay)
        ax.axis("off")
        ok = "✓" if pred["label"] == label else "✗"
        ax.set_title(f"true: {label}\npred: {pred['label']} {pred['confidence']:.0%} {ok}", fontsize=9)
plt.suptitle("Grad-CAM on random test scans")
plt.tight_layout()
plt.show()

## 7. Save the results

In [ ]:
SAMPLE_FILES = []
if ADD_SAMPLE_SCANS:
    os.makedirs("static/samples", exist_ok=True)
    for label in clf.labels:
        for path in [f for f, c in test_files if c == label]:
            img = inference.open_image(open(path, "rb").read())
            pred = clf.predict(img, with_heatmap=False)
            if not inference.check_input(img)[0] and pred["label"] == label and pred["confident"]:
                out = os.path.join("static", "samples", f"{label}_1.jpg")
                inference.thumbnail(img, 512).save(out, quality=90)
                SAMPLE_FILES.append(out)
                break
    print("Sample scans:", SAMPLE_FILES)

meta = json.load(open(os.path.join(EXPORT_DIR, "model_meta.json")))
MODEL_FILES = [os.path.join(EXPORT_DIR, f) for f in ("model_meta.json", meta["body"], meta["head_weights"])]
RUN_FILES = [os.path.join(RUN_DIR, f) for f in ("metrics.json", "history.csv", "training_curves.png", "confusion_matrix.png")]
print("Model files:", MODEL_FILES)

In [ ]:
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = os.path.join("/content/drive/MyDrive/mri-model-runs", os.path.basename(RUN_DIR))
    shutil.copytree(RUN_DIR, os.path.join(dest, "run"), dirs_exist_ok=True)
    for f in MODEL_FILES + SAMPLE_FILES:
        os.makedirs(os.path.join(dest, os.path.dirname(f)), exist_ok=True)
        shutil.copy(f, os.path.join(dest, f))
    print("Saved to", dest)
    print("To use it without pushing: copy the files under", os.path.join(dest, EXPORT_DIR),
          "into the repo's models/ folder and commit them.")

**Push to GitHub (optional).** Create a fine-grained token with *Contents: read and write* on this
repository, add it as a Colab secret named `GITHUB_TOKEN`, set `PUSH_TO_GITHUB = True` and run the cell.
Render redeploys once the branch is merged into the branch it deploys from.

In [ ]:
if PUSH_TO_GITHUB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    git = ["git", "-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}"]
    run(git + ["add", *MODEL_FILES, *SAMPLE_FILES])
    run(git + ["add", "-f", *RUN_FILES])  # training/runs/ is git-ignored; keep only the small reports
    run(git + ["commit", "-m", f"Retrain model: {ARCH}, test accuracy {metrics['test_accuracy']:.1%}"])
    remote = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "push", remote, f"HEAD:{BRANCH}"], capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, "***"))
    if result.returncode:
        raise RuntimeError("git push failed")
else:
    print("PUSH_TO_GITHUB is False: nothing pushed.")